# Finetuning data: compare to pretraining and basic preparation

In [1]:
import jsonlines
import itertools
import pandas as pd
from pprint import pprint

import datasets
from datasets import load_dataset

/home/harry/Desktop/prcx_01/xx_github/06_deep_learning/00_short_courses/04-finetuning-large-language-models/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Pretrained Dataset

In [2]:
# To fix the 'Dataset scripts are no longer supported' error, use the allenai/c4 path
# which supports the new parquet-based loading.

pretrained_dataset = load_dataset("allenai/c4", "en", split="train", streaming=True)

In [35]:
finetuning_dataset_name = "lamini/lamini_docs"
finetuning_dataset = load_dataset(finetuning_dataset_name)
print(finetuning_dataset)

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1260
    })
    test: Dataset({
        features: ['question', 'answer', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 140
    })
})


In [3]:
n = 5
print("Pretrained dataset:")
top_n = itertools.islice(pretrained_dataset, n)
for i in top_n:
  print(i)

Pretrained dataset:
{'text': 'Beginners BBQ Class Taking Place in Missoula!\nDo you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class for everyone who wants to get better with their culinary skills.\nHe will teach you everything you need to know to compete in a KCBS BBQ competition, including techniques, recipes, timelines, meat selection and trimming, plus smoker and fire information.\nThe cost to be in the class is $35 per person, and for spectators it is free. Included in the cost will be either a t-shirt or apron and you will be tasting samples of each meat that is prepared.', 'timestamp': '2019-04-25 12:57:54', 'url': 'https://klyq.com/beginners-bbq-class-taking-place-in-missoula/'}
{'text': 'Discussion in \'Mac OS X Lion (10.7)\' started by axboi87, Jan 20, 2012.\nI\'ve got a 500gb intern

### Contrast with company finetuning dataset 

In [4]:
from datasets import load_dataset
import pandas as pd

try:
    # Loading from Hugging Face datasets library
    dataset = load_dataset("lamini/lamini_docs")
    # Convert to pandas dataframe for consistency with the rest of your workflow
    instruction_dataset_df = pd.DataFrame(dataset["train"])
    print("Dataset loaded successfully using Hugging Face datasets library.")
    display(instruction_dataset_df.head())
except Exception as e:
    print(f"Could not load dataset via library: {e}")
    print("Please upload lamini_docs.jsonl to the files tab manually.")

Generating test split: 100%|██████████| 140/140 [00:00<00:00, 24299.71 examples/s]


Dataset loaded successfully using Hugging Face datasets library.


,question,answer,input_ids,attention_mask,labels
0,How can I evaluate the performance and quality...,There are several metrics that can be used to ...,"[2347, 476, 309, 7472, 253, 3045, 285, 3290, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[2347, 476, 309, 7472, 253, 3045, 285, 3290, 2..."
1,Can I find information about the code's approa...,"Yes, the code includes methods for submitting ...","[5804, 309, 1089, 1491, 670, 253, 2127, 434, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[5804, 309, 1089, 1491, 670, 253, 2127, 434, 2..."
2,How does Lamini AI handle requests for generat...,Lamini AI offers features for generating text ...,"[2347, 1057, 418, 4988, 74, 14980, 6016, 9762,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[2347, 1057, 418, 4988, 74, 14980, 6016, 9762,..."
3,Does the `submit_job()` function expose any ad...,It is unclear which `submit_job()` function is...,"[10795, 253, 2634, 21399, 64, 17455, 42702, 11...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[10795, 253, 2634, 21399, 64, 17455, 42702, 11..."
4,Does the `add_data()` function support differe...,"No, the `add_data()` function does not support...","[10795, 253, 2634, 1911, 64, 2203, 42702, 1159...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[10795, 253, 2634, 1911, 64, 2203, 42702, 1159..."


## Various ways of formatting your data

In [6]:
examples = instruction_dataset_df.to_dict()
examples

{'question': {0: 'How can I evaluate the performance and quality of the generated text from Lamini models?',
  1: "Can I find information about the code's approach to handling long-running tasks and background jobs?",
  2: 'How does Lamini AI handle requests for generating text that requires reasoning or decision-making based on given information?',
  3: 'Does the `submit_job()` function expose any advanced training options such as learning rate schedules or early stopping?',
  4: 'Does the `add_data()` function support different data augmentation techniques or preprocessing options for training data?',
  5: 'Can Lamini generate text for data storytelling or data visualization purposes?',
  6: 'Can the documentation predict the outcome of a coin toss?',
  7: 'How does the `submit_job()` function work in Lamini? What does it mean to submit a job, and what happens behind the scenes?',
  8: 'Does Lamini support generating code',
  9: 'Can Lamini be used to create chatbots or virtual assis

In [11]:
text = examples['question'][0] + '\n' + examples['answer'][0]
print(text)

How can I evaluate the performance and quality of the generated text from Lamini models?
There are several metrics that can be used to evaluate the performance and quality of generated text from Lamini models, including perplexity, BLEU score, and human evaluation. Perplexity measures how well the model predicts the next word in a sequence, while BLEU score measures the similarity between the generated text and a reference text. Human evaluation involves having human judges rate the quality of the generated text based on factors such as coherence, fluency, and relevance. It is recommended to use a combination of these metrics for a comprehensive evaluation of the model's performance.


In [12]:
if "question" in examples and "answer" in examples:
  text = examples["question"][0] + examples["answer"][0]
elif "instruction" in examples and "response" in examples:
  text = examples["instruction"][0] + examples["response"][0]
elif "input" in examples and "output" in examples:
  text = examples["input"][0] + examples["output"][0]
else:
  text = examples["text"][0]

In [13]:
prompt_template_qa = """### Question:
{question}

### Answer:
{answer}"""

In [20]:
question = examples["question"][0]
answer = examples["answer"][0]

text_with_prompt_template = prompt_template_qa.format(question=question, answer=answer)
print(text_with_prompt_template)

### Question:
How can I evaluate the performance and quality of the generated text from Lamini models?

### Answer:
There are several metrics that can be used to evaluate the performance and quality of generated text from Lamini models, including perplexity, BLEU score, and human evaluation. Perplexity measures how well the model predicts the next word in a sequence, while BLEU score measures the similarity between the generated text and a reference text. Human evaluation involves having human judges rate the quality of the generated text based on factors such as coherence, fluency, and relevance. It is recommended to use a combination of these metrics for a comprehensive evaluation of the model's performance.


In [24]:
prompt_template_q = """### Question:
{question}

### Answer:"""

In [25]:
num_examples = len(examples['question'])
num_examples

1260

In [26]:
finetuning_dataset_text_only = []
finetuning_dataset_question_answer = []

In [28]:
for i in range(num_examples):
  question = examples["question"][i]
  answer = examples["answer"][i]

  text_with_prompt_template_qa = prompt_template_qa.format(question=question, answer=answer)
  finetuning_dataset_text_only.append({"text": text_with_prompt_template_qa})

  text_with_prompt_template_q = prompt_template_q.format(question=question)
  finetuning_dataset_question_answer.append({"question": text_with_prompt_template_q, "answer": answer})

In [29]:
finetuning_dataset_question_answer

[{'question': '### Question:\nHow can I evaluate the performance and quality of the generated text from Lamini models?\n\n### Answer:',
  'answer': "There are several metrics that can be used to evaluate the performance and quality of generated text from Lamini models, including perplexity, BLEU score, and human evaluation. Perplexity measures how well the model predicts the next word in a sequence, while BLEU score measures the similarity between the generated text and a reference text. Human evaluation involves having human judges rate the quality of the generated text based on factors such as coherence, fluency, and relevance. It is recommended to use a combination of these metrics for a comprehensive evaluation of the model's performance."},
 {'question': "### Question:\nCan I find information about the code's approach to handling long-running tasks and background jobs?\n\n### Answer:",
  'answer': 'Yes, the code includes methods for submitting jobs, checking job status, and retrie

In [31]:
pprint(finetuning_dataset_text_only[0:5])

[{'text': '### Question:\n'
          'How can I evaluate the performance and quality of the generated '
          'text from Lamini models?\n'
          '\n'
          '### Answer:\n'
          'There are several metrics that can be used to evaluate the '
          'performance and quality of generated text from Lamini models, '
          'including perplexity, BLEU score, and human evaluation. Perplexity '
          'measures how well the model predicts the next word in a sequence, '
          'while BLEU score measures the similarity between the generated text '
          'and a reference text. Human evaluation involves having human judges '
          'rate the quality of the generated text based on factors such as '
          'coherence, fluency, and relevance. It is recommended to use a '
          'combination of these metrics for a comprehensive evaluation of the '
          "model's performance."},
 {'text': '### Question:\n'
          "Can I find information about the code's a

In [32]:
pprint(finetuning_dataset_question_answer[0])

{'answer': 'There are several metrics that can be used to evaluate the '
           'performance and quality of generated text from Lamini models, '
           'including perplexity, BLEU score, and human evaluation. Perplexity '
           'measures how well the model predicts the next word in a sequence, '
           'while BLEU score measures the similarity between the generated '
           'text and a reference text. Human evaluation involves having human '
           'judges rate the quality of the generated text based on factors '
           'such as coherence, fluency, and relevance. It is recommended to '
           'use a combination of these metrics for a comprehensive evaluation '
           "of the model's performance.",
 'question': '### Question:\n'
             'How can I evaluate the performance and quality of the generated '
             'text from Lamini models?\n'
             '\n'
             '### Answer:'}


### Common ways of storing your data

In [33]:
with jsonlines.open(f'lamini_docs_processed.jsonl', 'w') as writer:
    writer.write_all(finetuning_dataset_question_answer)

In [34]:
finetuning_dataset_name = "lamini/lamini_docs"
finetuning_dataset = load_dataset(finetuning_dataset_name)
print(finetuning_dataset)

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1260
    })
    test: Dataset({
        features: ['question', 'answer', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 140
    })
})
